# Track 1 (Notebook 03b — Parked Modules): Dataplex Discovery, Profiling, AI Insights, AutoDQ, Cloud DLP, Custom Aspects & Serverless FinOps
> **📌 Note**: These modules were parked from `03_Governance_Security_and_Data_Quality.ipynb` so that Notebook 03 focuses strictly on **Row-Level Security (RLS), Column-Level Security (CLS), and Dynamic Data Masking purely in BigQuery SQL**. This parked notebook can be discarded at the end of the workshop.

- **Module 1**: Cloud Storage Lakehouse Data Discovery Scan (`RFP C1.1.1.1`)
- **Module 2**: Automated Statistical Data Profiling (`RFP C1.1.1.4`)
- **Module 3**: AI Data Insights & Dataset Knowledge Graph (`RFP C1.1.1.10`)
- **Module 4**: Automated Data Quality (AutoDQ YAML) & Quarantine Ledger (`RFP C1.1.1.5`–`C1.1.1.7`)
- **Module 5**: Sensitive Data Protection (Cloud DLP) Built-in & Custom InfoType Scan (`RFP C1.1.5.3`)
- **Module 6**: Dataplex Custom Governance Aspect Types & AI-Automated Aspect Tagging (`RFP C1.1.1.8`)
- **Module 8**: Serverless FinOps Cost Attribution & Workload Telemetry (`RFP C1.1.1.18`, `C1.1.6.6`)

In [ ]:
# @title Step 0: Auto-Detect `PROJECT_ID` & `USER_EMAIL`, Create `acsm_observability` Dataset, and Enable Governance APIs
import os, shutil, subprocess, time, urllib.request

PROJECT_ID = ""  # @param {type:"string"}
LOCATION = "asia-southeast1"  # @param {type:"string"}

if not PROJECT_ID or PROJECT_ID == "<PROJECT_ID>":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT") or subprocess.check_output(["gcloud", "config", "get-value", "project"], text=True).strip()
USER_EMAIL = subprocess.check_output(["gcloud", "config", "get-value", "account"], text=True).strip()
os.environ.update({"PROJECT_ID": PROJECT_ID, "GOOGLE_CLOUD_PROJECT": PROJECT_ID, "LOCATION": LOCATION, "USER_EMAIL": USER_EMAIL})

%load_ext google.cloud.bigquery

for fname in ["governance_helper.py", "dq_rules_gold_customer360.yaml"]:
    if os.path.exists(f"../scripts/{fname}"):
        shutil.copy(f"../scripts/{fname}", fname)
    else:
        urllib.request.urlretrieve(
            f"https://raw.githubusercontent.com/arvind-dhariwal/aeon-credit-gcp-workshop/main/track1_platform_governance/scripts/{fname}?ts={int(time.time())}",
            fname,
        )

!gcloud config set project {PROJECT_ID} --quiet
!gcloud services enable dataplex.googleapis.com dlp.googleapis.com cloudaicompanion.googleapis.com aiplatform.googleapis.com datalineage.googleapis.com datacatalog.googleapis.com bigquerydatapolicy.googleapis.com bigquery.googleapis.com --project={PROJECT_ID} --quiet
!bq --location={LOCATION} mk --dataset --description="ACSM Platform Observability, Data Profile, AutoDQ Results, Quarantine, DLP Findings & FinOps Layer" {PROJECT_ID}:acsm_observability 2>/dev/null || true
!bq --location={LOCATION} mk --table {PROJECT_ID}:acsm_observability.dataplex_profile_scan_results 2>/dev/null || true
!bq --location={LOCATION} mk --table {PROJECT_ID}:acsm_observability.dataplex_dq_native_export 2>/dev/null || true
print(f"✅ Project: {PROJECT_ID} | User: {USER_EMAIL} | Region: {LOCATION} | Observability Dataset: {PROJECT_ID}.acsm_observability ready")

In [ ]:
# @title Module 1 & 2: Run Dataplex GCS Discovery Scan (`acsm-gcs-lakehouse-discovery-scan`) & Data Profiling Scan (`acsm-gold-customer360-profile-scan`)
!python3 governance_helper.py data-discovery --project={PROJECT_ID} --location={LOCATION}
!gcloud dataplex datascans create data-profile acsm-gold-customer360-profile-scan \
  --project={PROJECT_ID} --location={LOCATION} \
  --data-source-resource="//bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/acsm_gold/tables/gold_aeon_customer360_profile" \
  --export-results-table="//bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/acsm_observability/tables/dataplex_profile_scan_results" \
  --enable-catalog-publishing --on-demand=true 2>/dev/null || true
!gcloud dataplex datascans run acsm-gold-customer360-profile-scan --project={PROJECT_ID} --location={LOCATION} --quiet
!python3 governance_helper.py data-profile --project={PROJECT_ID} --location={LOCATION}

In [ ]:
# @title Module 3 & 4: Run AI Data Insights (`acsm_silver,acsm_gold`) & AutoDQ Scan (`dq_rules_gold_customer360.yaml`)
!python3 governance_helper.py data-insights --project={PROJECT_ID} --location={LOCATION} --datasets=acsm_silver,acsm_gold
!cat dq_rules_gold_customer360.yaml
!gcloud dataplex datascans create data-quality acsm-gold-customer360-quality-scan \
  --project={PROJECT_ID} --location={LOCATION} \
  --data-source-resource="//bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/acsm_gold/tables/gold_aeon_customer360_profile" \
  --data-quality-spec-file=dq_rules_gold_customer360.yaml \
  --export-results-table="//bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/acsm_observability/tables/dataplex_dq_native_export" \
  --enable-catalog-publishing --on-demand=true 2>/dev/null || true
!python3 governance_helper.py data-quality --project={PROJECT_ID} --location={LOCATION}

In [ ]:
# @title Module 5, 6 & 8: Run Cloud DLP Custom InfoType Scan, Gemini Custom Governance Aspects & Serverless FinOps Telemetry
!python3 governance_helper.py sdp-pii-scan --project={PROJECT_ID} --location={LOCATION}
!python3 governance_helper.py ai-catalog-governance --project={PROJECT_ID} --location={LOCATION} --user-email={USER_EMAIL}
!python3 governance_helper.py finops-telemetry --project={PROJECT_ID} --location={LOCATION}

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- Verify Parked Observability Tables (Discovery, Profile, AutoDQ, DLP, Custom Aspects & FinOps)
SELECT 'dataplex_discovery_scan_results' AS observability_table, COUNT(*) AS row_count FROM `acsm_observability.dataplex_discovery_scan_results`
UNION ALL
SELECT 'dataplex_profile_summary', COUNT(*) FROM `acsm_observability.dataplex_profile_summary`
UNION ALL
SELECT 'dataplex_dq_scan_results', COUNT(*) FROM `acsm_observability.dataplex_dq_scan_results`
UNION ALL
SELECT 'sdp_pii_findings', COUNT(*) FROM `acsm_observability.sdp_pii_findings`
UNION ALL
SELECT 'dataplex_ai_catalog_aspects', COUNT(*) FROM `acsm_observability.dataplex_ai_catalog_aspects`
UNION ALL
SELECT 'vw_finops_job_telemetry', COUNT(*) FROM `acsm_observability.vw_finops_job_telemetry`;